In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

warm_up_datavidia_arkavidia_path = kagglehub.competition_download('warm-up-datavidia-arkavidia')

print('Data source import complete.')


# Import Library

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb

# Configuration

In [ ]:
df_cams = pd.read_csv('/kaggle/input/competitions/warm-up-datavidia-arkavidia/release/cams_kualitas_udara.csv')
df_ispu = pd.read_csv('/kaggle/input/competitions/warm-up-datavidia-arkavidia/release/ispu_jakarta.csv')
df_test = pd.read_csv('/kaggle/input/competitions/warm-up-datavidia-arkavidia/release/test.csv')
df_train = pd.read_csv('/kaggle/input/competitions/warm-up-datavidia-arkavidia/release/train.csv')

In [ ]:
df_train = df_train.drop(columns='_is_test', errors='ignore')
df_test  = df_test.drop(columns='_is_test', errors='ignore')

In [ ]:
print(df_cams.info())
print(df_ispu.info())
print(df_test.info())
print(df_train.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1246 entries, 0 to 1245
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   tanggal                1246 non-null   datetime64[ns]
 1   pm10                   1246 non-null   float64       
 2   pm2_5                  1246 non-null   float64       
 3   carbon_monoxide        1246 non-null   float64       
 4   nitrogen_dioxide       1246 non-null   float64       
 5   sulphur_dioxide        1246 non-null   float64       
 6   ozone                  1246 non-null   float64       
 7   aerosol_optical_depth  1246 non-null   float64       
 8   dust                   1246 non-null   float64       
 9   uv_index               1246 non-null   float64       
 10  us_aqi                 1245 non-null   float64       
 11  european_aqi           1245 non-null   float64       
dtypes: datetime64[ns](1), float64(11)
memory usage: 116.9 KB
None


In [ ]:
df_cams.isnull().sum()

tanggal                  0
pm10                     0
pm2_5                    0
carbon_monoxide          0
nitrogen_dioxide         0
sulphur_dioxide          0
ozone                    0
aerosol_optical_depth    0
dust                     0
uv_index                 0
us_aqi                   1
european_aqi             1
dtype: int64

In [ ]:
import pandas as pd
import numpy as np

# ===== 1. Seragamkan skema =====
for name, df in [('train', df_train), ('test', df_test), ('ispu', df_ispu), ('cams', df_cams)]:
    df['tanggal'] = pd.to_datetime(df['tanggal'], errors='coerce')
    print(f"{name:5s} | tanggal gagal parse: {df['tanggal'].isna().sum()} | "
          f"rentang: {df['tanggal'].min().date()} s/d {df['tanggal'].max().date()}")

df_train = df_train.drop(columns='_is_test')
df_test  = df_test.drop(columns='_is_test')

# duplikat tanggal (ISPU wajar duplikat karena 5 stasiun)
print("dup train:", df_train['tanggal'].duplicated().sum(),
      "| dup cams:", df_cams['tanggal'].duplicated().sum())

# ===== 2. Audit target =====
print(df_train['pm25'].describe())
print("pm25 <= 0:", (df_train['pm25'] <= 0).sum(),
      "| pm25 > 300:", (df_train['pm25'] > 300).sum())
print(df_train.groupby(df_train['tanggal'].dt.year)['pm25']
      .agg(n_obs='count', n_missing=lambda s: s.isna().sum(), median='median'))

# ===== 3. Rapikan ISPU =====
print(df_ispu['jenis_data'].value_counts(dropna=False))
print(df_ispu['kode_stasiun'].value_counts(dropna=False))
print(df_ispu[df_ispu['kode_stasiun'].isna()])          # lihat baris aneh
df_ispu = df_ispu.dropna(subset=['kode_stasiun'])

ispu_daily = (df_ispu.groupby('tanggal')
              .agg(ispu_pm25_med=('pm25', 'median'),
                   ispu_pm25_max=('pm25', 'max'),
                   ispu_n_stasiun=('pm25', 'count'))
              .reset_index())

# ===== 4. Peta cakupan =====
cal = pd.DataFrame({'tanggal': pd.date_range('2021-01-01', '2025-12-31', freq='D')})
cov = (cal.merge(df_train[['tanggal', 'pm25']], on='tanggal', how='left')
          .merge(ispu_daily[['tanggal', 'ispu_pm25_med']], on='tanggal', how='left')
          .merge(df_cams[['tanggal', 'pm2_5', 'us_aqi']], on='tanggal', how='left'))

print("CAMS tanggal hilang:",
      pd.date_range(df_cams['tanggal'].min(), df_cams['tanggal'].max())
        .difference(df_cams['tanggal']).date.tolist())

miss = cov['pm25'].isna()
print(f"\nDari {miss.sum()} hari pm25 kosong:")
print("  ada ISPU pm25 :", cov.loc[miss, 'ispu_pm25_med'].notna().sum())
print("  ada CAMS      :", cov.loc[miss, 'pm2_5'].notna().sum())
print("  tidak ada apa2:", cov.loc[miss, ['ispu_pm25_med', 'pm2_5']].isna().all(axis=1).sum())

# ===== 5. Kelayakan sebagai pengisi =====
print("\nKorelasi Spearman dengan pm25 (periode overlap):")
print(cov[['pm25', 'ispu_pm25_med', 'pm2_5', 'us_aqi']].corr(method='spearman')['pm25'])

train | tanggal gagal parse: 0 | rentang: 2021-01-01 s/d 2025-12-31
test  | tanggal gagal parse: 0 | rentang: 2026-01-01 s/d 2026-03-31
ispu  | tanggal gagal parse: 0 | rentang: 2021-01-01 s/d 2025-11-30
cams  | tanggal gagal parse: 0 | rentang: 2022-08-04 s/d 2025-12-31


KeyError: "['_is_test'] not found in axis"